<a href="https://colab.research.google.com/github/saf-does-code/weather-datapipeline/blob/main/pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import requests

url = "https://api.open-meteo.com/v1/forecast?latitude=25.07&longitude=55.14&daily=temperature_2m_max&timezone=Asia%2FDubai"
response = requests.get(url)
print(response.json())


{'latitude': 25.06151, 'longitude': 55.183193, 'generationtime_ms': 0.02372264862060547, 'utc_offset_seconds': 14400, 'timezone': 'Asia/Dubai', 'timezone_abbreviation': 'GMT+4', 'elevation': -7.0, 'daily_units': {'time': 'iso8601', 'temperature_2m_max': '°C'}, 'daily': {'time': ['2026-06-16', '2026-06-17', '2026-06-18', '2026-06-19', '2026-06-20', '2026-06-21', '2026-06-22'], 'temperature_2m_max': [37.2, 40.5, 40.6, 38.3, 37.5, 37.8, 38.1]}}


In [5]:
data = response.json()

dates = data['daily']['time']
temps = data['daily']['temperature_2m_max']

for date, temp in zip(dates, temps):
    print(f"{date} → {temp}°C")

2026-06-16 → 37.2°C
2026-06-17 → 40.5°C
2026-06-18 → 40.6°C
2026-06-19 → 38.3°C
2026-06-20 → 37.5°C
2026-06-21 → 37.8°C
2026-06-22 → 38.1°C


In [6]:
import csv

filename = "weather_dubai.csv"

with open(filename, "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["date", "temperature_max_c"])
    for date, temp in zip(dates, temps):
        writer.writerow([date, temp])

print(f"Saved {len(dates)} rows to {filename}")

Saved 7 rows to weather_dubai.csv


In [7]:
import psycopg2

conn = psycopg2.connect("YOUR_CONNECTION_STRING_HERE")
cursor = conn.cursor()

for date, temp in zip(dates, temps):
    cursor.execute("""
        INSERT INTO weather (date, temperature_max_c)
        VALUES (%s, %s)
        ON CONFLICT (date) DO NOTHING
    """, (date, temp))

conn.commit()
cursor.close()
conn.close()

print("Data inserted successfully!")

Data inserted successfully!
